# 04: Neural Networks & CNNs with PyTorch

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the building blocks of neural networks (neurons, layers, activations)
2. **Implement** a simple neural network from scratch with NumPy
3. **Use PyTorch** to define, train, and evaluate neural networks
4. **Build CNNs** (Convolutional Neural Networks) for image classification
5. **Apply transfer learning** using pretrained models

## Prerequisites

- Comfortable with NumPy and basic linear algebra
- Familiar with basic ML concepts (supervised learning, loss functions, gradient descent)
- Python class syntax (we'll use `nn.Module` extensively)

---

**Roadmap:**

| Section | Topic |
|---------|-------|
| 1 | The Neuron — building block of neural nets |
| 2 | From Scratch — a 2-layer network in NumPy |
| 3 | PyTorch Basics — tensors, autograd, nn.Module |
| 4 | First PyTorch Model — classify the moons dataset |
| 5 | Training Loop Anatomy — understanding every step |
| 6 | MNIST with Fully Connected Networks |
| 7 | Convolutional Neural Networks (CNNs) |
| 8 | Transfer Learning |
| 9 | Practical Tips & Exercises |

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

from shared.viz_utils import setup_style, save_fig
setup_style()

# Automatically detect the best available device
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---

## Section 1: The Neuron — Building Block of Neural Networks

### What is a Neuron?

A single artificial neuron performs two operations:

1. **Linear combination**: $z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$
2. **Activation**: $a = \sigma(z)$

The **weights** ($w$) control how much each input matters. The **bias** ($b$) shifts the decision boundary. The **activation function** ($\sigma$) introduces non-linearity.

### Why Non-Linearity?

Without activation functions, stacking layers collapses to a single linear transformation:

$$\text{Layer 2}(\text{Layer 1}(\mathbf{x})) = W_2(W_1\mathbf{x} + b_1) + b_2 = (W_2 W_1)\mathbf{x} + (W_2 b_1 + b_2) = W'\mathbf{x} + b'$$

That's just **one** linear layer! Non-linear activations let neural networks learn complex, non-linear decision boundaries.

### Common Activation Functions

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| **Sigmoid** | $\frac{1}{1+e^{-z}}$ | (0, 1) | Output layer for binary classification |
| **Tanh** | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | (-1, 1) | Hidden layers (zero-centered) |
| **ReLU** | $\max(0, z)$ | [0, inf) | Default for hidden layers (fast, effective) |
| **Leaky ReLU** | $\max(0.01z, z)$ | (-inf, inf) | Avoids "dying ReLU" problem |

In [ ]:
# --- Visualize Activation Functions and Their Derivatives ---

z = np.linspace(-5, 5, 200)

# Define activations and their derivatives
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh(x):
    return np.tanh(x)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def leaky_relu(x, alpha=0.1):
    return np.where(x > 0, x, alpha * x)

def leaky_relu_deriv(x, alpha=0.1):
    return np.where(x > 0, 1.0, alpha)

activations = [
    ("Sigmoid", sigmoid, sigmoid_deriv),
    ("Tanh", tanh, tanh_deriv),
    ("ReLU", relu, relu_deriv),
    ("Leaky ReLU", leaky_relu, leaky_relu_deriv),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, (name, func, deriv) in enumerate(activations):
    # Activation
    axes[0, i].plot(z, func(z), linewidth=2, color='steelblue')
    axes[0, i].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, i].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, i].set_title(f"{name}", fontsize=13, fontweight='bold')
    axes[0, i].set_xlabel("z")
    if i == 0:
        axes[0, i].set_ylabel("Activation f(z)")
    
    # Derivative
    axes[1, i].plot(z, deriv(z), linewidth=2, color='coral')
    axes[1, i].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, i].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, i].set_title(f"{name} Derivative", fontsize=13)
    axes[1, i].set_xlabel("z")
    if i == 0:
        axes[1, i].set_ylabel("Derivative f'(z)")

plt.suptitle("Activation Functions & Their Derivatives", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, "activation_functions")
plt.show()

print("Key observations:")
print("- Sigmoid/Tanh derivatives vanish for large |z| --> 'vanishing gradient' problem")
print("- ReLU derivative is 0 for z<0 --> 'dying ReLU' (neuron stops learning)")
print("- Leaky ReLU fixes dying ReLU by allowing small gradients for z<0")
print("- ReLU is the most common default for hidden layers")

---

## Section 2: From Scratch — A Simple Neural Network in NumPy

### The XOR Problem

XOR (exclusive or) is a classic problem that **cannot** be solved by a single-layer network (it's not linearly separable). This was famously pointed out by Minsky & Papert in 1969 and contributed to the first "AI winter."

| x1 | x2 | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 0  | 1  | 1   |
| 1  | 0  | 1   |
| 1  | 1  | 0   |

A 2-layer network CAN solve it. Let's prove this by building one from scratch.

### Network Architecture
```
Input (2) --> Hidden (4, sigmoid) --> Output (1, sigmoid)
```

We'll implement:
1. **Forward pass**: compute predictions
2. **Loss**: binary cross-entropy
3. **Backward pass**: compute gradients via chain rule
4. **Update**: gradient descent

In [ ]:
# --- Neural Network from Scratch: XOR ---

np.random.seed(42)

# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
y = np.array([[0], [1], [1], [0]], dtype=np.float64)

# Sigmoid (reuse from above)
def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv_np(z):
    s = sigmoid_np(z)
    return s * (1 - s)

# --- First, show that a SINGLE layer FAILS ---
print("=" * 50)
print("Attempt 1: Single-Layer Network (will fail!)")
print("=" * 50)

# Single layer: Input(2) -> Output(1)
W_single = np.random.randn(2, 1) * 0.5
b_single = np.zeros((1, 1))
lr = 1.0
losses_single = []

for epoch in range(2000):
    # Forward
    z = X @ W_single + b_single
    a = sigmoid_np(z)
    
    # Loss (binary cross-entropy)
    loss = -np.mean(y * np.log(a + 1e-8) + (1 - y) * np.log(1 - a + 1e-8))
    losses_single.append(loss)
    
    # Backward
    dz = a - y  # derivative of BCE + sigmoid combined
    dW = X.T @ dz / len(X)
    db = np.mean(dz, axis=0, keepdims=True)
    
    # Update
    W_single -= lr * dW
    b_single -= lr * db

predictions_single = (sigmoid_np(X @ W_single + b_single) > 0.5).astype(int)
print(f"Final loss: {losses_single[-1]:.4f}")
print(f"Predictions: {predictions_single.flatten()} (target: [0, 1, 1, 0])")
print(f"Single layer CANNOT learn XOR!\n")

# --- Now, a 2-layer network that WORKS ---
print("=" * 50)
print("Attempt 2: Two-Layer Network (will succeed!)")
print("=" * 50)

# Architecture: Input(2) -> Hidden(4) -> Output(1)
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))
lr = 2.0
losses_twolayer = []

for epoch in range(2000):
    # --- Forward pass ---
    z1 = X @ W1 + b1           # (4, 4)
    a1 = sigmoid_np(z1)         # hidden activations
    z2 = a1 @ W2 + b2           # (4, 1)
    a2 = sigmoid_np(z2)         # output prediction
    
    # --- Loss ---
    loss = -np.mean(y * np.log(a2 + 1e-8) + (1 - y) * np.log(1 - a2 + 1e-8))
    losses_twolayer.append(loss)
    
    # --- Backward pass (chain rule!) ---
    # Output layer gradients
    dz2 = a2 - y                          # (4, 1)
    dW2 = a1.T @ dz2 / len(X)            # (4, 1)
    db2 = np.mean(dz2, axis=0, keepdims=True)
    
    # Hidden layer gradients (chain rule through sigmoid)
    da1 = dz2 @ W2.T                      # (4, 4)
    dz1 = da1 * sigmoid_deriv_np(z1)      # element-wise
    dW1 = X.T @ dz1 / len(X)             # (2, 4)
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    # --- Update weights ---
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

# Final predictions
hidden = sigmoid_np(X @ W1 + b1)
output = sigmoid_np(hidden @ W2 + b2)
predictions_twolayer = (output > 0.5).astype(int)
print(f"Final loss: {losses_twolayer[-1]:.4f}")
print(f"Predictions: {predictions_twolayer.flatten()} (target: [0, 1, 1, 0])")
print(f"Raw outputs: {output.flatten().round(3)}")
print(f"Two-layer network SOLVES XOR!")

In [ ]:
# --- Plot loss curves: single vs two-layer ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_single, color='crimson', linewidth=2)
axes[0].set_title("Single-Layer Network (FAILS)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (BCE)")
axes[0].set_ylim(0, 1.0)
axes[0].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Random guess')
axes[0].legend()

axes[1].plot(losses_twolayer, color='seagreen', linewidth=2)
axes[1].set_title("Two-Layer Network (SUCCEEDS)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss (BCE)")
axes[1].set_ylim(0, 1.0)
axes[1].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Random guess')
axes[1].legend()

plt.suptitle("XOR Problem: Why Depth Matters", fontsize=15, fontweight='bold')
plt.tight_layout()
save_fig(fig, "xor_loss_comparison")
plt.show()

print("The single-layer network plateaus at ~0.69 (random guess loss = ln(2))")
print("The two-layer network drives loss close to 0 -- it learned the XOR pattern!")

---

## Section 3: PyTorch Basics

Writing backpropagation by hand is educational but painful. **PyTorch** handles it automatically!

### The Big Three Concepts

1. **Tensors**: Like NumPy arrays, but can run on GPUs and track gradients
2. **Autograd**: Automatic differentiation — PyTorch builds a computation graph and computes gradients for you
3. **nn.Module**: Base class for all neural network models — organize your parameters and define `forward()`

### The 3-Step Pattern for Every PyTorch Project

```python
# Step 1: Define the model
model = MyNetwork()

# Step 2: Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Step 3: Training loop
for epoch in range(num_epochs):
    predictions = model(X)          # Forward pass
    loss = criterion(predictions, y) # Compute loss
    optimizer.zero_grad()            # Clear old gradients
    loss.backward()                  # Compute new gradients
    optimizer.step()                 # Update weights
```

In [ ]:
# --- PyTorch Tensors & Autograd ---

# Creating tensors
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.randn(3, 3)  # random normal

print("Basic tensors:")
print(f"  a = {a}, shape = {a.shape}, dtype = {a.dtype}")
print(f"  b = {b.shape}, c = {c.shape}")

# NumPy <-> PyTorch
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)
back_to_np = tensor_from_np.numpy()
print(f"\nNumPy -> Tensor -> NumPy roundtrip: {back_to_np}")

# GPU movement (if available)
tensor_on_device = a.to(device)
print(f"Tensor on {device}: {tensor_on_device}")

# --- Autograd: automatic differentiation ---
print("\n--- Autograd Demo ---")
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()  # y = x1^2 + x2^2 + x3^2

print(f"x = {x.data}")
print(f"y = x1^2 + x2^2 + x3^2 = {y.item()}")

y.backward()  # Compute dy/dx

print(f"dy/dx = 2x = {x.grad}")  # Should be [2, 4, 6]
print("\nPyTorch computed the gradients automatically!")

# --- A more complex example ---
print("\n--- Chain Rule Demo ---")
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x + 1       # y = 3x + 1
z = y ** 2           # z = (3x + 1)^2
z.backward()         # dz/dx = 2(3x+1) * 3 = 6(3*2+1) = 42
print(f"x = {x.item()}, z = (3x+1)^2 = {z.item()}")
print(f"dz/dx = 6(3x+1) = {x.grad.item()} (expected: 42)")

---

## Section 4: First PyTorch Model — Moons Classification

Let's build our first real PyTorch model! The **moons dataset** is a classic 2D binary classification problem with a non-linear decision boundary.

We'll follow the 3-step pattern:
1. Define a model (`nn.Module`)
2. Choose loss function + optimizer
3. Write the training loop

In [ ]:
# --- Generate and visualize the moons dataset ---
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_moons, y_moons = make_moons(n_samples=1000, noise=0.2, random_state=42)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', 
                     alpha=0.7, edgecolors='k', linewidths=0.5, s=30)
ax.set_title("Moons Dataset — Non-Linearly Separable", fontsize=14, fontweight='bold')
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.colorbar(scatter, label="Class")
plt.tight_layout()
plt.show()

print(f"Dataset: {X_moons.shape[0]} samples, {X_moons.shape[1]} features, 2 classes")
print("A straight line CANNOT separate these classes. We need a neural network!")

In [ ]:
# --- Step 1: Define the model ---

class MoonNet(nn.Module):
    """Simple feedforward network for binary classification."""
    
    def __init__(self, input_dim=2, hidden1=32, hidden2=16):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),  # single output for binary classification
        )
    
    def forward(self, x):
        return self.network(x)

# Create model and inspect it
model = MoonNet().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")
print("(That's 2*32 + 32 + 32*16 + 16 + 16*1 + 1 = 64 + 32 + 512 + 16 + 16 + 1 = 641)")

In [ ]:
# --- Steps 2 & 3: Loss, optimizer, and training loop ---

# Prepare data
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42
)

X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1).to(device)

# Step 2: Loss and optimizer
criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy (includes sigmoid)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Step 3: Training loop
losses = []
accuracies = []

for epoch in range(200):
    model.train()
    
    # Forward pass
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    
    # Backward pass
    optimizer.zero_grad()  # Clear accumulated gradients
    loss.backward()        # Compute gradients
    optimizer.step()       # Update weights
    
    # Track metrics
    losses.append(loss.item())
    
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == y_train_t).float().mean().item()
        accuracies.append(acc)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Train Acc: {acc:.4f}")

# Evaluate on test set
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_preds = (torch.sigmoid(test_logits) > 0.5).float()
    test_acc = (test_preds == y_test_t).float().mean().item()
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
# --- Visualize: training curves + decision boundary ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(losses, color='steelblue', linewidth=2)
axes[0].set_title("Training Loss", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")

# Accuracy curve
axes[1].plot(accuracies, color='seagreen', linewidth=2)
axes[1].set_title("Training Accuracy", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.5, 1.05)

# Decision boundary
h = 0.02  # mesh step size
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

grid_tensor = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)
model.eval()
with torch.no_grad():
    Z = torch.sigmoid(model(grid_tensor)).cpu().numpy().reshape(xx.shape)

axes[2].contourf(xx, yy, Z, levels=50, cmap='coolwarm', alpha=0.6)
axes[2].contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
axes[2].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm',
               edgecolors='k', linewidths=0.5, s=40)
axes[2].set_title(f"Decision Boundary (Test Acc: {test_acc:.1%})", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Feature 1")
axes[2].set_ylabel("Feature 2")

plt.suptitle("Moons Classification with PyTorch", fontsize=15, fontweight='bold')
plt.tight_layout()
save_fig(fig, "moons_classification")
plt.show()

print("The neural network learned a smooth, non-linear decision boundary!")

---

## Section 5: The Training Loop Anatomy

Let's break down exactly what happens in each step of the training loop:

```python
for epoch in range(num_epochs):
    # 1. FORWARD PASS: data flows through the network
    predictions = model(X)
    
    # 2. COMPUTE LOSS: how wrong are we?
    loss = criterion(predictions, y)
    
    # 3. ZERO GRADIENTS: clear old gradients (PyTorch accumulates by default!)
    optimizer.zero_grad()
    
    # 4. BACKWARD PASS: compute gradients via chain rule (autograd)
    loss.backward()
    
    # 5. UPDATE WEIGHTS: take a step in the direction that reduces loss
    optimizer.step()
```

### Why `optimizer.zero_grad()`?

PyTorch **accumulates** gradients by default. If you forget `zero_grad()`, gradients from previous batches pile up, and your training goes haywire!

This design choice exists because gradient accumulation is sometimes intentional (e.g., to simulate larger batch sizes). But 99% of the time, you want to zero them out before each backward pass.

### `model.train()` vs `model.eval()`

- `model.train()`: enables dropout and batch normalization in training mode
- `model.eval()`: disables dropout, uses running stats for batch norm
- **Always** switch modes when going from training to evaluation!

### `torch.no_grad()`

During evaluation, we don't need gradients. Wrapping code in `with torch.no_grad():` saves memory and computation.

---

## Section 6: MNIST with a Fully Connected Network

Time for a real-world dataset! **MNIST** contains 70,000 handwritten digits (28x28 grayscale images, classes 0-9). It's the "Hello World" of deep learning.

We'll:
1. Load MNIST using `torchvision`
2. Visualize some samples
3. Build a fully connected (FC) network: 784 -> 256 -> 128 -> 10
4. Train with mini-batch DataLoaders
5. Evaluate with confusion matrix

In [ ]:
# --- Load MNIST ---

transform = transforms.Compose([
    transforms.ToTensor(),               # Convert to tensor and scale to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

# DataLoaders handle batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Image shape:      {train_dataset[0][0].shape} (channels, height, width)")
print(f"Number of batches (train): {len(train_loader)}")
print(f"Classes: 0-9 (digits)")

In [ ]:
# --- Visualize sample MNIST images ---

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}", fontsize=10)
    ax.axis('off')

plt.suptitle("MNIST Sample Images", fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, "mnist_samples")
plt.show()

In [ ]:
# --- Define the Fully Connected MNIST Network ---

class MNISTFullyConnected(nn.Module):
    """Fully connected network for MNIST classification.
    
    Architecture: 784 -> 256 -> 128 -> 10
    Each image (28x28 = 784 pixels) is flattened into a 1D vector.
    """
    
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()  # 28x28 -> 784
        self.fc_layers = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # 10 classes (digits 0-9)
        )
    
    def forward(self, x):
        x = self.flatten(x)      # (batch, 1, 28, 28) -> (batch, 784)
        x = self.fc_layers(x)    # (batch, 784) -> (batch, 10)
        return x                  # raw logits (CrossEntropyLoss applies softmax)

fc_model = MNISTFullyConnected().to(device)
print(fc_model)
print(f"\nTotal parameters: {sum(p.numel() for p in fc_model.parameters()):,}")

In [ ]:
# --- Train the FC model on MNIST ---

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(fc_model.parameters(), lr=0.001)

num_epochs = 5
fc_train_losses = []
fc_train_accs = []

for epoch in range(num_epochs):
    fc_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = fc_model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    fc_train_losses.append(epoch_loss)
    fc_train_accs.append(epoch_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

In [ ]:
# --- Evaluate the FC model ---
from sklearn.metrics import confusion_matrix, classification_report

fc_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = fc_model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
fc_accuracy = (all_preds == all_labels).mean()

print(f"FC Network Test Accuracy: {fc_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=3))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title(f"FC Network Confusion Matrix (Acc: {fc_accuracy:.1%})", fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, "mnist_fc_confusion_matrix")
plt.show()

In [ ]:
# --- Show some misclassified examples ---

misclassified_idx = np.where(all_preds != all_labels)[0]
print(f"Total misclassified: {len(misclassified_idx)} out of {len(all_labels)}")

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for i, ax in enumerate(axes.flat):
    if i < len(misclassified_idx):
        idx = misclassified_idx[i]
        image, _ = test_dataset[idx]
        ax.imshow(image.squeeze(), cmap='gray')
        ax.set_title(f"True: {all_labels[idx]}\nPred: {all_preds[idx]}", 
                     fontsize=9, color='red', fontweight='bold')
    ax.axis('off')

plt.suptitle("Misclassified Examples — Can You See Why the Model Was Confused?", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, "mnist_misclassified")
plt.show()

---

## Section 7: Convolutional Neural Networks (CNNs)

### Why CNNs?

The FC network above **flattens** each 28x28 image into a 784-length vector. This throws away all spatial structure! A pixel at position (0,0) and position (27,27) are treated the same way.

**CNNs** preserve spatial relationships by using **convolutional filters** that slide across the image, detecting local patterns (edges, textures, shapes).

### Key CNN Building Blocks

| Layer | What it does |
|-------|-------------|
| `Conv2d` | Applies learnable filters to detect features (edges, patterns) |
| `ReLU` | Non-linear activation (same as before) |
| `MaxPool2d` | Downsamples by taking the max in each region (reduces size, adds invariance) |
| `BatchNorm2d` | Normalizes activations (stabilizes training, allows higher learning rates) |
| `Dropout` | Randomly zeros some activations during training (prevents overfitting) |
| `Flatten` | Converts 2D feature maps to 1D vector for the final FC layers |

### Typical CNN Architecture Pattern

```
Input Image
    |
[Conv2d -> ReLU -> MaxPool2d]   <-- Extract low-level features (edges)
    |
[Conv2d -> ReLU -> MaxPool2d]   <-- Extract higher-level features (shapes)
    |
[Flatten]                       <-- Reshape for classification
    |
[Linear -> ReLU -> Dropout]     <-- Classify based on features
    |
[Linear -> Output]              <-- Final predictions
```

### Convolution Intuition

A 3x3 filter slides across the image, computing dot products at each position:

```
Image patch:     Filter:        Output:
[1 0 1]         [1  0  1]
[0 1 0]    *    [0  1  0]    =  1+0+1+0+1+0+0+0+1 = 4
[1 0 1]         [0  0  1]
```

The network **learns** the filter values during training!

In [ ]:
# --- Define the CNN for MNIST ---

class SimpleCNN(nn.Module):
    """CNN for MNIST digit classification.
    
    Architecture:
        Conv(1->32, 3x3) -> ReLU -> MaxPool(2x2)    -> 32 x 14 x 14
        Conv(32->64, 3x3) -> ReLU -> MaxPool(2x2)   -> 64 x 7 x 7
        Flatten                                       -> 3136
        Linear(3136->128) -> ReLU -> Dropout(0.5)
        Linear(128->10)
    """
    
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 1x28x28 -> 32x28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 32x28x28 -> 32x14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # 32x14x14 -> 64x14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 64x14x14 -> 64x7x7
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                                  # 64x7x7 -> 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

cnn_model = SimpleCNN().to(device)
print(cnn_model)

# Compare parameter counts
fc_params = sum(p.numel() for p in fc_model.parameters())
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\nFC Network parameters:  {fc_params:>10,}")
print(f"CNN parameters:         {cnn_params:>10,}")
print(f"\nCNN uses {'fewer' if cnn_params < fc_params else 'more'} parameters but will be MORE accurate!")

In [ ]:
# --- Train the CNN on MNIST ---

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

num_epochs = 5
cnn_train_losses = []
cnn_train_accs = []

for epoch in range(num_epochs):
    cnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    cnn_train_losses.append(epoch_loss)
    cnn_train_accs.append(epoch_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

# Evaluate CNN on test set
cnn_model.eval()
cnn_preds = []
cnn_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        _, predicted = outputs.max(1)
        cnn_preds.extend(predicted.cpu().numpy())
        cnn_labels.extend(labels.cpu().numpy())

cnn_preds = np.array(cnn_preds)
cnn_labels = np.array(cnn_labels)
cnn_accuracy = (cnn_preds == cnn_labels).mean()

print(f"\n{'='*50}")
print(f"FC Network Test Accuracy:  {fc_accuracy:.4f}")
print(f"CNN Test Accuracy:         {cnn_accuracy:.4f}")
print(f"Improvement:               {(cnn_accuracy - fc_accuracy)*100:+.2f}%")
print(f"{'='*50}")

In [ ]:
# --- Compare FC vs CNN: training curves ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)

# Loss comparison
axes[0].plot(epochs_range, fc_train_losses, 'o-', color='coral', linewidth=2, label='FC Network')
axes[0].plot(epochs_range, cnn_train_losses, 's-', color='steelblue', linewidth=2, label='CNN')
axes[0].set_title("Training Loss: FC vs CNN", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_xticks(list(epochs_range))

# Accuracy comparison
axes[1].plot(epochs_range, fc_train_accs, 'o-', color='coral', linewidth=2, label=f'FC (test: {fc_accuracy:.1%})')
axes[1].plot(epochs_range, cnn_train_accs, 's-', color='steelblue', linewidth=2, label=f'CNN (test: {cnn_accuracy:.1%})')
axes[1].set_title("Training Accuracy: FC vs CNN", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_ylim(0.9, 1.01)
axes[1].set_xticks(list(epochs_range))

plt.suptitle("MNIST: Fully Connected vs Convolutional Network", fontsize=15, fontweight='bold')
plt.tight_layout()
save_fig(fig, "fc_vs_cnn_comparison")
plt.show()

print("The CNN achieves higher accuracy by exploiting the spatial structure of images!")

---

## Section 8: Transfer Learning

### The Key Insight

Training a deep CNN from scratch requires:
- A **large** dataset (millions of images)
- A **lot** of compute (days on GPUs)
- Careful tuning of architecture and hyperparameters

But models like **ResNet**, **VGG**, and **EfficientNet** have already been trained on ImageNet (14 million images, 1000 classes). They've learned general-purpose visual features:
- Early layers: edges, textures, colors
- Middle layers: shapes, patterns, object parts
- Late layers: high-level, task-specific features

### Transfer Learning Strategy

1. **Load** a pretrained model (e.g., ResNet18)
2. **Freeze** all pretrained layers (don't update their weights)
3. **Replace** the final classification layer with one matching your number of classes
4. **Train** only the new layer (fast, needs little data)
5. Optionally **fine-tune**: unfreeze some later layers and train with a very small learning rate

### When to Use Transfer Learning

| Your dataset | Similar to ImageNet? | Strategy |
|-------------|---------------------|----------|
| Small | Yes | Freeze all, train classifier only |
| Small | No | Freeze early layers, fine-tune later layers |
| Large | Yes | Fine-tune the whole network |
| Large | No | Train from scratch (or fine-tune cautiously) |

In [ ]:
# --- Transfer Learning Demo: ResNet18 ---

# Load pretrained ResNet18
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

# Step 1: Freeze all pretrained parameters
for param in resnet.parameters():
    param.requires_grad = False

# Step 2: Replace the final classification layer
# Original: Linear(512, 1000) for ImageNet's 1000 classes
# New:      Linear(512, 10) for our 10-class task
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 10)  # Only this layer will be trained

# Move to device
resnet = resnet.to(device)

# Count parameters
total_params = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("ResNet18 Transfer Learning Setup")
print("=" * 50)
print(f"Total parameters:     {total_params:>12,}")
print(f"Frozen parameters:    {frozen_params:>12,} ({frozen_params/total_params:.1%})")
print(f"Trainable parameters: {trainable_params:>12,} ({trainable_params/total_params:.1%})")
print(f"\nWe only need to train {trainable_params:,} out of {total_params:,} parameters!")
print(f"That's {total_params/trainable_params:.0f}x less work than training from scratch.")

print(f"\n--- Final Layer (the only trainable part) ---")
print(f"resnet.fc = {resnet.fc}")
print(f"\n(We're not training this model here — save it for the project!)")

In [ ]:
# --- Visualize the ResNet18 architecture (high-level) ---

print("ResNet18 Architecture Overview")
print("=" * 60)

layer_info = []
for name, module in resnet.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    status = "TRAINABLE" if trainable > 0 else "frozen"
    layer_info.append((name, type(module).__name__, params, status))
    print(f"  {name:12s} | {type(module).__name__:20s} | {params:>9,} params | {status}")

print("=" * 60)
print("\nThe pretrained conv/bn/layer blocks extract features.")
print("We only replace and train the final 'fc' layer for our task.")

---

## Section 9: Practical Tips for Training Neural Networks

### Getting Started
- **Start simple**: begin with a small network and add complexity only if needed
- **Verify your pipeline**: overfit on a tiny subset first (if the model can't memorize 10 samples, something is broken)
- **Use established architectures**: don't invent new architectures unless you have a good reason

### Monitoring Training
- **Track both training AND validation loss** every epoch
- **Overfitting** = training loss decreases but validation loss increases
- **Underfitting** = both losses are high and not decreasing
- Use **early stopping**: stop training when validation loss stops improving

### Hyperparameters
- **Learning rate** is the single most important hyperparameter
  - Too high: loss oscillates or diverges
  - Too low: training is painfully slow
  - Try: 0.001 (Adam default), then adjust by 3-10x up/down
- **Batch size**: 32-128 is a good starting point
- **Number of epochs**: train until validation loss plateaus

### Regularization (Preventing Overfitting)
- **Dropout**: randomly zero out neurons during training (typical: 0.2-0.5)
- **BatchNorm**: normalize layer outputs (stabilizes training, acts as mild regularization)
- **Data augmentation**: for images — rotations, flips, crops, color jitter
- **Weight decay**: L2 regularization on weights (set via optimizer: `weight_decay=1e-4`)

### Saving and Loading Models
```python
# Save the best model
torch.save(model.state_dict(), 'best_model.pth')

# Load it later
model = MyCNN()
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
```

---

## Key Takeaways

1. **Neural networks** = stacked linear transformations + non-linear activations. Without activations, deep networks collapse to a single linear layer.

2. **The PyTorch pattern** is always the same:
   - Define model as `nn.Module` with `__init__` and `forward`
   - Choose loss function (`CrossEntropyLoss`, `BCEWithLogitsLoss`, etc.)
   - Choose optimizer (`Adam`, `SGD`, etc.)
   - Training loop: forward -> loss -> zero_grad -> backward -> step

3. **CNNs exploit spatial structure** in images. Convolutional layers detect local patterns (edges, textures, shapes) and are far more effective than fully connected networks for image tasks.

4. **Transfer learning**: don't reinvent the wheel. Pretrained models like ResNet have already learned powerful visual features. Freeze layers, replace the classifier, and fine-tune.

5. **Start simple, monitor metrics, iterate**. The most common mistakes are overcomplicating the model too early and not watching for overfitting.

---

## Exercises

### Exercise 1: Improve the FC MNIST Network

Modify the fully connected MNIST network:
- Add `BatchNorm1d` after each hidden layer (before ReLU)
- Try different hidden layer sizes (e.g., 512 -> 256 -> 128)
- Add Dropout(0.3) after each ReLU

Does accuracy improve? How does training speed change?

In [ ]:
# Exercise 1: Improved FC MNIST Network
# TODO: Complete this model

class ImprovedFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc_layers = nn.Sequential(
            # TODO: Add layers with BatchNorm and Dropout
            # Hint: nn.Linear(784, 512),
            #       nn.BatchNorm1d(512),
            #       nn.ReLU(),
            #       nn.Dropout(0.3),
            #       ... more layers ...
            #       nn.Linear(..., 10),
            pass
        )
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc_layers(x)
        return x

# TODO: Train and compare accuracy with the original FC model
# improved_model = ImprovedFC().to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(improved_model.parameters(), lr=0.001)
# ... training loop ...

### Exercise 2: Data Augmentation for CNN

Add data augmentation to the MNIST CNN training pipeline. Use:
- `transforms.RandomRotation(10)` — rotate up to 10 degrees
- `transforms.RandomAffine(0, translate=(0.1, 0.1))` — shift up to 10%

Create augmented and non-augmented DataLoaders. Train the CNN on each for 5 epochs and compare test accuracy.

Does augmentation help? (Hint: MNIST is already quite clean, so the effect may be small. It matters more on harder datasets.)

In [ ]:
# Exercise 2: Data Augmentation
# TODO: Complete this exercise

# Define augmented transform
# augmented_transform = transforms.Compose([
#     transforms.RandomRotation(10),
#     transforms.RandomAffine(0, translate=(0.1, 0.1)),
#     transforms.ToTensor(),
#     transforms.Normalize((0.1307,), (0.3081,))
# ])

# Load augmented dataset
# aug_train_dataset = torchvision.datasets.MNIST(
#     root='./data', train=True, download=True, transform=augmented_transform
# )
# aug_train_loader = DataLoader(aug_train_dataset, batch_size=64, shuffle=True)

# TODO: Train a new SimpleCNN() on the augmented data for 5 epochs
# TODO: Compare test accuracy with the non-augmented CNN
# TODO: Visualize some augmented images to see what the transforms do

### Exercise 3: Fashion-MNIST with a CNN

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) is a drop-in replacement for MNIST with clothing items instead of digits. It's harder because the classes look more similar to each other.

Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

Use the same `SimpleCNN` architecture. Load Fashion-MNIST with:
```python
torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
```

Train for 5-10 epochs. What accuracy do you get? How does it compare to regular MNIST?

In [ ]:
# Exercise 3: Fashion-MNIST
# TODO: Complete this exercise

# Load Fashion-MNIST
# fashion_train = torchvision.datasets.FashionMNIST(
#     root='./data', train=True, download=True, transform=transform
# )
# fashion_test = torchvision.datasets.FashionMNIST(
#     root='./data', train=False, download=True, transform=transform
# )
# fashion_train_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)
# fashion_test_loader = DataLoader(fashion_test, batch_size=256, shuffle=False)

# Class names for Fashion-MNIST
# fashion_classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
#                    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# TODO: Visualize some Fashion-MNIST samples
# TODO: Create a new SimpleCNN() and train it
# TODO: Evaluate and show confusion matrix
# TODO: Which classes are most often confused?

---

## Next Steps

In the upcoming project notebook, you will apply everything from this notebook to build a complete image classification pipeline:
- Load and preprocess a real dataset
- Build and train a CNN from scratch
- Apply transfer learning with a pretrained model
- Compare approaches and analyze results

**Concepts to review before the project:**
- The PyTorch training loop (Section 5)
- CNN architecture design (Section 7)
- Transfer learning setup (Section 8)
- Practical tips for training (Section 9)